In [19]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [20]:
from google import genai
from google.genai import types
client = genai.Client()

In [21]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [22]:
print("How many lesson pages are in the dataset?")
print(len(documents))

How many lesson pages are in the dataset?
72


In [23]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

In [24]:
print("Q2. indexing and searching")
question = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(question)
search_results

Q2. indexing and searching


[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [25]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''

In [26]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['content'])

    return '\n'.join(lines).strip()

In [27]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [28]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
How does the agentic loop keep calling the model until it stops?

Context:
# The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools

In [29]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)
print(response.text)

Based on the provided text, the agentic loop keeps calling the model until it stops through a specific mechanism that relies on the model's response and a `has_function_calls` flag.

Here's a step-by-step breakdown:

1.  **Initialization:**
    *   The loop starts with a `while True` statement, indicating it will run indefinitely until an explicit `break` condition is met.
    *   A `has_function_calls` flag is initialized to `False` at the beginning of each iteration.
    *   The `messages` list, which represents the conversation history, is continuously updated. It initially contains the `developer` instructions and the `user`'s question.

2.  **Model Call:**
    *   Inside the loop, the `openai_client.responses.create` method is called. This is the point where the LLM (`gpt-5.4-mini`) is invoked.
    *   Crucially, the entire `messages` history is passed as `input` to the model in every iteration. This allows the model to "remember" the previous turns, including its own previous fun

In [30]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Based on the provided text, the agentic loop keeps calling the model until it stops through a specific mechanism that relies on the model's response and a `has_function_calls` flag.

Here's a step-by-step breakdown:

1.  **Initialization:**
    *   The loop starts with a `while True` statement, indicating it will run indefinitely until an explicit `break` condition is met.
    *   A `has_function_calls` flag is initialized to `False` at the beginning of each iteration.
    *   The `messages` list, which represents the conversation history, is continuously updated. It initially contains the `developer` instructions and the `user`'s question.

2.  **Model Call:**
    *   Inside the loop, the `openai_client.responses.create` method is called. This is the point where the LLM (`gpt-5.4-mini`) is invoked.
    *   Crucially,

The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: chunk_documents. It uses a sliding window - a window of size characters slides across the text in steps of step characters, and each window position becomes one chunk:

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print("Q4. How many chunks do you get?")
print(len(chunks))

Q4. How many chunks do you get?
295


In [14]:
index.fit(chunks)

In [15]:
question = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(question)
search_results

[{'start': 4000,
  'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model rea

In [16]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
How does the agentic loop keep calling the model until it stops?

Context:
while` loop. The loop keeps calling the model until
it returns a response without any function calls. We also keep an
iteration counter so we can see how many round-trips happened.

```python
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break
```

This is the core agent 

In [17]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)
print(response.text)

The agentic loop keeps calling the model until it stops through a combination of a continuous `while True` loop, memory, a feedback mechanism, and a specific exit condition.

Here's a breakdown:

1.  **Continuous Calling (`while True`):**
    *   The core of the loop is `while True:`, which means the code inside the loop will execute indefinitely unless explicitly stopped.
    *   In each iteration, the `openai_client.responses.create()` function is called, sending the current `messages` history to the model and requesting a new response. This is the mechanism for *keeping* the model called.

2.  **Memory and Feedback Loop:**
    *   **Memory:** The `messages` list serves as the agent's memory. After each model call, `messages.extend(response.output)` adds the model's latest output (which can include its own messages or requested function calls) to this history.
    *   **Tool Execution & Input for Next Turn:** If the model's `response.output` contains items of `type == "function_call"

In [18]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""The agentic loop keeps calling the model until it stops through a combination of a continuous `while True` loop, memory, a feedback mechanism, and a specific exit condition.

Here's a breakdown:

1.  **Continuous Calling (`while True`):**
    *   The core of the loop is `while True:`, which means the code inside the loop will execute indefinitely unless explicitly stopped.
    *   In each iteration, the `openai_client.responses.create()` function is called, sending the current `messages` history to the model and requesting a new response. This is the mechanism for *keeping* the model called.

2.  **Memory and Feedback Loop:**
    *   **Memory:** The `messages` list serves as the agent's memory. After each model call, `messages.extend(response.output)` adds the model's latest output (which can include its own messages 

In [37]:
INSTRUCTIONS = '''
You are a senior tech lead. Use the search_knowledge_base tool to 
retrieve facts before answering technical questions. If the tool 
returns no results, do not make up an answer.
'''

In [33]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

# 1. Define the LangChain Tool
@tool
def search_knowledge_base(query: str) -> str:
    """
    Searches the internal knowledge base for technical documentation and guidelines.
    Always use this tool to find factual answers from our internal documents.
    """
    # Assuming 'index' is your pre-fitted minsearch.Index or AppendableIndex 
    print(f"🔧 Agent searching minsearch for: {query}")
    
    # Query minsearch (adjust num_results or filters based on your setup)
    search_results = index.search(query=query, num_results=3)
    
    if not search_results:
        return "No relevant information found in the knowledge base."
        
    # LangChain tools expect a single string return. 
    # We map the dict results to a formatted text block.
    # Adjust the "text" key if your minsearch text_fields uses a different primary name.
    formatted_results = "\n\n".join([
        f"--- Document {i+1} ---\n{res.get('text', '')}" 
        for i, res in enumerate(search_results)
    ])
    
    return formatted_results

In [41]:
from langchain_core.prompts import ChatPromptTemplate

# 2. Initialize Gemini 2.5 Flash via LangChain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2 # Keep temperature low to prevent tool-use hallucinations
)

# 4. Construct the Agent
tools = [search_knowledge_base]


# 3. Create the Agent Prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        '''
        You are a senior tech lead. Use the search_knowledge_base tool to 
        retrieve facts before answering technical questions. If the tool 
        returns no results, do not make up an answer.
        '''
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)

# 5. Create the AgentExecutor
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, # Set to True so you can watch the agent's thought process
    max_iterations=5 # Hard cap to prevent infinite search loops
)

In [42]:
# Test a scenario where the agent is forced to use the minsearch tool
response = agent_executor.invoke({"input": "How does the agentic loop work, and how is it different from plain RAG?"})
print("\nFinal Answer:\n", response["output"])



> Entering new AgentExecutor chain...

Invoking: `search_knowledge_base` with `{'query': 'agentic loop vs plain RAG'}`


🔧 Agent searching minsearch for: agentic loop vs plain RAG
--- Document 1 ---


--- Document 2 ---


--- Document 3 ---
[{'type': 'text', 'text': "I'm sorry, but I couldn't find any information in my knowledge base about how the agentic loop works or how it differs from plain RAG.", 'extras': {'signature': 'CiQBDDnWx9KTwwhQe4FjYp+YdgnXSZBBhJLmipSrKJQCZBX8gPkKaQEMOdbH1y7C5ENyILPrKBXYzdgBDFLCIfJtpbEHxctjDGQF+xSAAHnzpcwJv7+RKkmuEsSSJOcF875hYXv4X2bQMWvE591EQ6UebjieyAKObauTsy5cWLglIC9A7nn61oegJuZc3vHS7QqJAgEMOdbHnKap3UCKZzFnhl7Rg67jCwizPeuS5/ZNexMEnAQvwLXzUJJ6qNqEqnOv2YO5JYlInqXkHljxyDkaLIDZK3jzbmz8HsbXq3mj2YgV1E4xQtGvAB4RRj0Ml+l3ScTCA+Xr4XIEouhb0HF3RD3tUVQOs+gQkOrNkP+Emjy9OI4tGT07jyQQpGOuK20TRqiaT13Qc90XRtKqAXGvvAlm5qBN9x6oAREK7EFIW2oBywjgKjlnKwvXDHXGSTEZnnAjeTss2iDHulTjxmb6OVBEqGHCzPZKQ3i3XHWnc/Oi44haIY1hjDdYyES64cwygLqctjzT4fCukx72I+ljIXgL5whxmO2wliIKUwEMOdbHcSMxPev8